In [ ]:
import pandas as pd
import numpy as np
import os

def escape_latex(text):
    """LaTeX 문법과 충돌하는 특수문자들을 이스케이프 처리하고 줄바꿈을 변환합니다."""
    if pd.isna(text) or text == "":
        return ""

    text = str(text)

    # 이스케이프해야 할 특수 문자들
    chars_to_escape = {
        '%': '\\%',
        '&': '\\&',
        '$': '\\$',
        '_': '\\_',
        '#': '\\#',
        '{': '\\{',
        '}': '\\}'
    }
    for char, escaped_char in chars_to_escape.items():
        text = text.replace(char, escaped_char)

    # 엑셀의 줄바꿈(Alt+Enter)을 LaTeX의 줄바꿈으로 변경
    text = text.replace('\n', '\\\\ ')
    return text

def convert_excel_to_latex(excel_file, output_file):
    if not os.path.exists(excel_file):
        print(f"오류: '{excel_file}' 파일을 찾을 수 없습니다. 파이썬 스크립트와 같은 폴더에 파일을 놔주세요.")
        return

    print("엑셀 파일을 읽는 중입니다...")
    # 엑셀 파일 읽기
    df = pd.read_excel(excel_file, dtype=str)

    latex_output = ""

    for index, row in df.iterrows():
        try:
            # 별점을 제외한 4가지 변수만 수집
            date = escape_latex(row.get('작성일자', ''))
            user_id = escape_latex(row.get('작성자아이디', ''))

            # 객실명은 없는 경우(NaN) 빈 문자열로 처리
            room = row.get('객실명', '')
            room_name = escape_latex(room) if pd.notna(room) else ""

            content = escape_latex(row.get('리뷰내용', ''))

        except Exception as e:
            print(f"데이터 변환 중 {index+1}번째 줄에서 오류 발생: {e}")
            continue

        # 4개의 인자만 들어가는 LaTeX 포맷에 맞게 문자열 조립
        latex_block = f"\\glampingReview{{{date}}}\n{{{user_id}}}\n{{{room_name}}}\n{{{content}}}\n\n"
        latex_output += latex_block

    # 결과물 txt 파일로 저장
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(latex_output)

    print(f"\n변환 완료! 총 {len(df)}개의 리뷰가 '{output_file}' 파일로 생성되었습니다.")
    print("생성된 파일의 내용을 복사하여 LaTeX 코드의 \\section*{...} 아래에 붙여넣으세요.")

if __name__ == "__main__":
    EXCEL_FILENAME = 'reviews.xlsx'
    OUTPUT_FILENAME = 'reviews_latex_code.txt'

    convert_excel_to_latex(EXCEL_FILENAME, OUTPUT_FILENAME)